# 07 — Anomaly Signal Backtesting

**Goal:** Close the loop — did the confirmed anomaly signals from notebook 04 actually predict *anything useful*?

## Strategy
When a confirmed anomaly is detected on day T, we **buy at the next open (day T+1)** and hold for N days (N = 5, 10, 20). We then compare forward returns against the **buy-and-hold baseline** for the same period.

This answers the fundamental analyst question: *"Our model flagged these dates — but do they have predictive value?"*

**DuckDB SQL** is the primary engine for:
- Building the trade log
- Computing per-trade P&L
- Aggregating win rates, average returns, and risk-adjusted stats by holding period
- Comparing anomaly trades vs random-entry baseline

In [1]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.rcParams.update({
    'figure.dpi'       : 150,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.22,
    'grid.linestyle'   : ':',
})

DATA_DIR = Path('data')
OUT_DIR  = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

COLORS = {
    'AAPL': '#1565C0', 'AMZN': '#E65100', 'GLD' : '#2E7D32',
    'JPM' : '#C62828', 'SPY' : '#6A1B9A', 'XOM' : '#4E342E'
}
HOLDING_PERIODS = [5, 10, 20]   # days to hold after signal
print('Imports OK')

Imports OK


In [2]:
# ── 2. Load data ───────────────────────────────────────────────────────────
df_price   = pd.read_csv(DATA_DIR / 'market_data.csv',     parse_dates=['Date'])
df_anomaly = pd.read_csv(DATA_DIR / 'anomaly_results.csv', parse_dates=['Date'])

df_price   = df_price.sort_values(['Ticker','Date']).reset_index(drop=True)
df_anomaly = df_anomaly.sort_values(['Ticker','Date']).reset_index(drop=True)

print(f'Price rows   : {len(df_price)}')
print(f'Anomaly rows : {len(df_anomaly)}')
print(f'Confirmed anomalies: {df_anomaly["IsConfirmed"].sum()}')

Price rows   : 12426
Anomaly rows : 12246
Confirmed anomalies: 74


## Build the Trade Log

For each confirmed anomaly date T and each holding period N, we look up:
- **Entry price**: Close on day T (anomaly day)
- **Exit price**: Close N trading days later
- **Forward return**: (exit - entry) / entry

We also build a **random baseline** by sampling the same number of trades from random non-anomaly dates, giving a fair comparison.

In [3]:
# ── 3. Build forward-return lookup table ─────────────────────────────────
# For each ticker, compute the N-day forward return for every date
trade_frames = []

for ticker in sorted(df_price['Ticker'].unique()):
    prices = df_price[df_price['Ticker'] == ticker][['Date','Close']].copy()
    prices = prices.sort_values('Date').reset_index(drop=True)

    for n in HOLDING_PERIODS:
        prices[f'fwd_{n}d'] = (
            prices['Close'].shift(-n) / prices['Close'] - 1
        ) * 100

    prices['Ticker'] = ticker
    trade_frames.append(prices)

fwd_df = pd.concat(trade_frames, ignore_index=True)

# Merge with anomaly flags
anomaly_flags = df_anomaly[['Date','Ticker','IsConfirmed']].copy()
fwd_df = fwd_df.merge(anomaly_flags, on=['Date','Ticker'], how='left')
fwd_df['IsConfirmed'] = fwd_df['IsConfirmed'].fillna(0).astype(int)

print(f'Forward return table: {fwd_df.shape}')
print(f'  Confirmed anomaly rows: {fwd_df["IsConfirmed"].sum()}')

Forward return table: (12426, 7)
  Confirmed anomaly rows: 74


## SQL Block 1 — Trade Results by Holding Period

For each ticker and holding period, compare:
- Average forward return on anomaly entry days
- Average forward return on random (non-anomaly) days
- Win rate (% of trades with positive return)

In [4]:
# ── 4. DuckDB — anomaly vs baseline returns per holding period ────────────
con = duckdb.connect()
con.register('fwd', fwd_df)

results_5d = con.execute("""
    SELECT
        Ticker,
        IsConfirmed                                          AS signal,
        COUNT(*)                                             AS n_trades,
        ROUND(AVG(fwd_5d),  2)                               AS avg_ret_5d_pct,
        ROUND(AVG(fwd_10d), 2)                               AS avg_ret_10d_pct,
        ROUND(AVG(fwd_20d), 2)                               AS avg_ret_20d_pct,
        ROUND(100.0 * SUM(CASE WHEN fwd_5d  > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate_5d,
        ROUND(100.0 * SUM(CASE WHEN fwd_10d > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate_10d,
        ROUND(100.0 * SUM(CASE WHEN fwd_20d > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate_20d
    FROM fwd
    WHERE fwd_5d IS NOT NULL
    GROUP BY Ticker, IsConfirmed
    ORDER BY Ticker, IsConfirmed DESC
""").df()

print('=== Anomaly (1) vs Random (0) — Forward Returns by Holding Period ===')
print(results_5d.to_string(index=False))

=== Anomaly (1) vs Random (0) — Forward Returns by Holding Period ===
Ticker  signal  n_trades  avg_ret_5d_pct  avg_ret_10d_pct  avg_ret_20d_pct  win_rate_5d  win_rate_10d  win_rate_20d
  AAPL       1        14            0.57             1.14             4.98         50.0          57.1          71.4
  AAPL       0      2052            0.52             1.05             2.11         57.6          59.6          62.8
  AMZN       1        16           -0.10            -1.10            -1.01         62.5          31.3          50.0
  AMZN       0      2050            0.41             0.81             1.56         54.9          58.1          60.8
   GLD       1        11            1.17             0.57             3.48         72.7          63.6          63.6
   GLD       0      2055            0.31             0.63             1.31         56.7          57.4          58.6
   JPM       1        13           -1.69            -0.99             1.62         53.8          53.8          38.5
  

## SQL Block 2 — Edge Summary (Anomaly Return MINUS Baseline)

The **edge** is the additional return an anomaly entry generates over a random entry. Positive edge = the signal adds value.

In [5]:
# ── 5. DuckDB — signal edge (anomaly return - baseline) ──────────────────
edge = con.execute("""
    WITH grouped AS (
        SELECT
            Ticker, IsConfirmed,
            AVG(fwd_5d)  AS avg_5d,
            AVG(fwd_10d) AS avg_10d,
            AVG(fwd_20d) AS avg_20d
        FROM fwd
        WHERE fwd_5d IS NOT NULL
        GROUP BY Ticker, IsConfirmed
    ),
    anom AS (SELECT * FROM grouped WHERE IsConfirmed = 1),
    base AS (SELECT * FROM grouped WHERE IsConfirmed = 0)
    SELECT
        a.Ticker,
        ROUND(a.avg_5d  - b.avg_5d,  3)  AS edge_5d_pct,
        ROUND(a.avg_10d - b.avg_10d, 3)  AS edge_10d_pct,
        ROUND(a.avg_20d - b.avg_20d, 3)  AS edge_20d_pct
    FROM anom a JOIN base b ON a.Ticker = b.Ticker
    ORDER BY edge_10d_pct DESC
""").df()

print('=== Signal Edge: Anomaly Entry Return - Baseline Return (%) ===')
print(edge.to_string(index=False))
print('\nPositive edge = anomaly signal generates BETTER returns than random entry.')

=== Signal Edge: Anomaly Entry Return - Baseline Return (%) ===
Ticker  edge_5d_pct  edge_10d_pct  edge_20d_pct
  AAPL        0.041         0.092         2.873
   GLD        0.863        -0.066         2.176
   JPM       -2.070        -1.726         0.203
  AMZN       -0.509        -1.907        -2.568
   XOM       -3.064        -2.058         1.838
   SPY       -1.074        -3.762        -1.834

Positive edge = anomaly signal generates BETTER returns than random entry.


## SQL Block 3 — SPY-Focused Trade Log

For SPY specifically, we build a full trade-by-trade log with entry price, exit price, and P&L — the kind of output you would show in a real trading analysis.

In [6]:
# ── 6. DuckDB — SPY trade log (20-day hold) ───────────────────────────────
spy_trades = con.execute("""
    SELECT
        Date                                              AS entry_date,
        ROUND(Close, 2)                                   AS entry_price,
        ROUND(Close * (1 + fwd_20d/100), 2)               AS exit_price_approx,
        ROUND(fwd_20d,  2)                                AS ret_20d_pct,
        CASE WHEN fwd_20d > 0 THEN 'WIN' ELSE 'LOSS' END  AS outcome
    FROM fwd
    WHERE Ticker = 'SPY'
      AND IsConfirmed = 1
      AND fwd_20d IS NOT NULL
    ORDER BY entry_date
""").df()

print(f'=== SPY Anomaly Trade Log (20-day hold) — {len(spy_trades)} trades ===')
print(spy_trades.to_string(index=False))
wins  = (spy_trades['outcome'] == 'WIN').sum()
losses = len(spy_trades) - wins
print(f'\n  Wins: {wins} ({wins/len(spy_trades)*100:.0f}%)   Losses: {losses}')
print(f'  Average return: {spy_trades["ret_20d_pct"].mean():.2f}%')
print(f'  Best trade   : {spy_trades["ret_20d_pct"].max():.2f}%')
print(f'  Worst trade  : {spy_trades["ret_20d_pct"].min():.2f}%')

=== SPY Anomaly Trade Log (20-day hold) — 14 trades ===
entry_date  entry_price  exit_price_approx  ret_20d_pct outcome
2018-10-10       248.15             250.57         0.97     WIN
2018-12-26       220.79             238.37         7.96     WIN
2019-08-05       256.92             263.18         2.44     WIN
2020-02-24       294.65             204.94       -30.44    LOSS
2020-02-27       271.88             240.11       -11.69    LOSS
2020-03-09       250.61             243.47        -2.85    LOSS
2020-06-11       276.33             293.23         6.11     WIN
2020-09-03       318.89             309.46        -2.96    LOSS
2021-01-27       348.55             355.92         2.12     WIN
2024-07-24       530.08             549.07         3.58     WIN
2024-12-18       575.96             594.43         3.21     WIN
2025-04-03       530.62             560.34         5.60     WIN
2025-04-09       542.40             558.66         3.00     WIN
2025-10-10       649.32             667.17      

## SQL Block 4 — Cumulative P&L vs Buy-and-Hold

We simulate $10,000 invested per confirmed anomaly signal and compare the equity curve to a simple SPY buy-and-hold strategy of the same capital.

In [7]:
# -- 7. Build equity curves ------------------------------------------------
INITIAL_CAPITAL = 10_000
HOLD_DAYS       = 20

spy_price = (df_price[df_price['Ticker'] == 'SPY'][['Date','Close']]
             .sort_values('Date').reset_index(drop=True))
spy_anom  = (df_anomaly[
    (df_anomaly['Ticker'] == 'SPY') & (df_anomaly['IsConfirmed'] == 1)
][['Date']].copy().reset_index(drop=True))
spy_anom  = spy_anom.merge(spy_price, on='Date').sort_values('Date').reset_index(drop=True)

# Position-index map for correct forward-price lookup in full series
price_idx = {date: i for i, date in enumerate(spy_price['Date'])}
prices    = spy_price['Close'].values

def get_exit_price(entry_date, hold):
    idx = price_idx.get(entry_date)
    if idx is None or idx + hold >= len(prices):
        return float('nan')
    return prices[idx + hold]

spy_anom['exit_price'] = spy_anom['Date'].apply(lambda d: get_exit_price(d, HOLD_DAYS))
spy_anom['fwd_ret']    = (spy_anom['exit_price'] / spy_anom['Close']) - 1
spy_anom = spy_anom.dropna(subset=['fwd_ret']).reset_index(drop=True)

spy_anom['trade_pnl']  = INITIAL_CAPITAL * spy_anom['fwd_ret']
spy_anom['cum_capital'] = INITIAL_CAPITAL + spy_anom['trade_pnl'].cumsum()

bnh = spy_price.copy()
bnh['bnh_capital'] = INITIAL_CAPITAL * bnh['Close'] / bnh['Close'].iloc[0]

print(f'Anomaly strategy trades: {len(spy_anom)}')
if len(spy_anom) > 0:
    print(f'Final capital (anomaly): ${spy_anom["cum_capital"].iloc[-1]:,.0f}')
print(f'Final capital (BnH)    : ${bnh["bnh_capital"].iloc[-1]:,.0f}')


Anomaly strategy trades: 14
Final capital (anomaly): $8,980
Final capital (BnH)    : $26,715


In [8]:
# ── 8. SQL Block 5 — trade stats summary ─────────────────────────────────
con.register('spy_trades_tbl', spy_anom)

trade_stats = con.execute("""
    SELECT
        COUNT(*)                                            AS total_trades,
        SUM(CASE WHEN fwd_ret > 0 THEN 1 ELSE 0 END)       AS wins,
        SUM(CASE WHEN fwd_ret < 0 THEN 1 ELSE 0 END)       AS losses,
        ROUND(100.0*SUM(CASE WHEN fwd_ret>0 THEN 1 ELSE 0 END)/COUNT(*), 1) AS win_rate_pct,
        ROUND(AVG(fwd_ret)*100,   2)                        AS avg_trade_ret_pct,
        ROUND(MAX(fwd_ret)*100,   2)                        AS best_trade_pct,
        ROUND(MIN(fwd_ret)*100,   2)                        AS worst_trade_pct,
        ROUND(STDDEV(fwd_ret)*100, 2)                       AS std_trade_ret_pct,
        ROUND(SUM(trade_pnl), 2)                            AS total_pnl_usd
    FROM spy_trades_tbl
""").df()

print('=== SPY Anomaly Strategy — Trade Summary (DuckDB) ===')
print(trade_stats.T.to_string(header=False))

=== SPY Anomaly Strategy — Trade Summary (DuckDB) ===
total_trades         14.00
wins                 10.00
losses                4.00
win_rate_pct         71.40
avg_trade_ret_pct    -0.73
best_trade_pct        7.96
worst_trade_pct     -30.44
std_trade_ret_pct     9.83
total_pnl_usd     -1020.11


## Strategy vs Buy-and-Hold Chart

Visually comparing the anomaly-entry equity curve against a simple SPY buy-and-hold. Each red dot marks a trade entry.

In [9]:
# ── 9. Equity curve chart ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False,
                                gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.35})

# Top: equity curves
ax1.plot(bnh['Date'], bnh['bnh_capital'],
         color='#BDBDBD', linewidth=1.5, label='SPY Buy-and-Hold', zorder=2)
ax1.step(spy_anom['Date'], spy_anom['cum_capital'],
         color='#1565C0', linewidth=2.0, where='post',
         label=f'Anomaly Strategy ({HOLD_DAYS}-day hold)', zorder=3)
ax1.scatter(spy_anom['Date'], spy_anom['cum_capital'],
            color='red', s=50, zorder=5, label='Trade Entry')
ax1.axhline(INITIAL_CAPITAL, color='black', linewidth=0.7, linestyle=':')
ax1.set_title(f'SPY Anomaly Strategy vs Buy-and-Hold\n'
              f'$10,000 starting capital | {HOLD_DAYS}-day holding period',
              fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Portfolio Value ($)', fontsize=10)
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.xaxis.set_major_locator(mdates.YearLocator())

# Bottom: per-trade return bar chart
colors = ['#2E7D32' if r > 0 else '#C62828' for r in spy_anom['fwd_ret']]
ax2.bar(range(len(spy_anom)), spy_anom['fwd_ret'] * 100, color=colors, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.6)
ax2.set_title(f'Per-Trade Return (%) — {len(spy_anom)} trades', fontsize=11, pad=8)
ax2.set_xlabel('Trade #', fontsize=10)
ax2.set_ylabel('Return (%)', fontsize=10)

plt.savefig(OUT_DIR / 'backtest_equity_curve.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/backtest_equity_curve.png')
plt.show()

Saved → outputs/backtest_equity_curve.png


## SQL Block 6 — All-Ticker Strategy Summary

Final ranked table: which ticker's anomaly signals had the best forward-return edge across all holding periods?

In [10]:
# ── 10. DuckDB — all-ticker edge ranked summary ───────────────────────────
all_ticker_edge = con.execute("""
    WITH anom_avg AS (
        SELECT Ticker,
               AVG(fwd_5d)  AS a5,
               AVG(fwd_10d) AS a10,
               AVG(fwd_20d) AS a20,
               COUNT(*)     AS n_signals
        FROM fwd
        WHERE IsConfirmed = 1 AND fwd_5d IS NOT NULL
        GROUP BY Ticker
    ),
    base_avg AS (
        SELECT Ticker,
               AVG(fwd_5d)  AS b5,
               AVG(fwd_10d) AS b10,
               AVG(fwd_20d) AS b20
        FROM fwd
        WHERE IsConfirmed = 0 AND fwd_5d IS NOT NULL
        GROUP BY Ticker
    )
    SELECT
        a.Ticker,
        a.n_signals,
        ROUND(a.a5,  2)          AS avg_ret_5d,
        ROUND(a.a10, 2)          AS avg_ret_10d,
        ROUND(a.a20, 2)          AS avg_ret_20d,
        ROUND(a.a5  - b.b5,  3)  AS edge_5d,
        ROUND(a.a10 - b.b10, 3)  AS edge_10d,
        ROUND(a.a20 - b.b20, 3)  AS edge_20d,
        -- simple composite score: average edge across periods
        ROUND((a.a5-b.b5 + a.a10-b.b10 + a.a20-b.b20) / 3, 3) AS composite_edge
    FROM anom_avg a JOIN base_avg b ON a.Ticker = b.Ticker
    ORDER BY composite_edge DESC
""").df()

print('=== All-Ticker Anomaly Signal Edge (ranked by composite) ===')
print(all_ticker_edge.to_string(index=False))

=== All-Ticker Anomaly Signal Edge (ranked by composite) ===
Ticker  n_signals  avg_ret_5d  avg_ret_10d  avg_ret_20d  edge_5d  edge_10d  edge_20d  composite_edge
  AAPL         14        0.57         1.14         4.98    0.041     0.092     2.873           1.002
   GLD         11        1.17         0.57         3.48    0.863    -0.066     2.176           0.991
   XOM          6       -2.72        -1.39         3.15   -3.064    -2.058     1.838          -1.095
   JPM         13       -1.69        -0.99         1.62   -2.070    -1.726     0.203          -1.198
  AMZN         16       -0.10        -1.10        -1.01   -0.509    -1.907    -2.568          -1.661
   SPY         14       -0.80        -3.19        -0.73   -1.074    -3.762    -1.834          -2.223


In [11]:
# ── 11. Save backtest results ─────────────────────────────────────────────
fwd_df[fwd_df['IsConfirmed'] == 1].to_csv(
    DATA_DIR / 'backtest_trades.csv', index=False
)
print('\n✓ 07_backtest complete')
print(f'  → data/backtest_trades.csv')
print(f'  → outputs/backtest_equity_curve.png')


✓ 07_backtest complete
  → data/backtest_trades.csv
  → outputs/backtest_equity_curve.png
